Publication Visuals and Analytics - BC
-----

# ----- [ CELLS ] ---------

## Define RUN TAG

In [ ]:
weather_year='2024'
run_date='20260603'
scenario_name='BASELINE'

- Tested tweaks on Plot Adjustments

In [ ]:
MARKER_SCALE_EXISTING=0.08
MARKER_SCALE_COMMITTED=0.08
MARKER_HIGHLIGHT_WIDTH=4
ANNOTATION_ADJUSTMENT_METER:float=30E3 

- Plot and data aggregation

In [ ]:
AGGREGATION_LEVEL='Province'

- Set Root

In [ ]:
from pathlib import Path
current_dir=Path.cwd()
# Go up 2 folders
root = current_dir.parent.parent
print('Root:', root)

paper_resources=current_dir
print('Paper Resources:', paper_resources)

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import RES.visuals as vis
from RES import utility as utils
from RES.hdf5_handler import DataHandler
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity
plt.style.use(root / 'RES' / 'visual_styles' / 'elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)

- Load Configs

In [ ]:
RUN_ID = f'{scenario_name.upper()}_{weather_year}_{run_date}'
config_name=f'CAN_{scenario_name.lower()}.yaml'

In [ ]:
cfg=utils.load_config(root / 'config' / config_name)
# run_id:str=cfg.get('Scenario').get('run_id')
country_name:str=cfg.get('country')
country_kwd=country_name.replace(' ','')
region_code='BC'

In [ ]:
sub_national_unit_tag:str=cfg.get('GADM').get('datafield_mapping').get('NAME_2',None) or cfg.get('GADM').get('datafield_mapping').get('NAME_1')

CRS_m = CRS_m = cfg.get('region_mapping').get(region_code).get('CRS_meters') or cfg.get('default_CRS').get('meters')  # Default metric CRS
CRS_d = cfg.get('default_CRS').get('degrees')  # Default geographic CRS
# regions=list(cfg.get('region_mapping').keys()) #'AL','BA','XK','ME','MK','RS'
regions=['BC']
region_to_code = {
    v["name"].replace(" ", ""): k
    for k, v in cfg.get("region_mapping").items()
}

vis_save_to_root=utils.ensure_path(paper_resources / f"vis/{RUN_ID}")
results_save_to_root=utils.ensure_path(paper_resources / f"results/{RUN_ID}")

## Load Store

In [ ]:
 #All the regions should have RUN_ID results available
combined_store:dict[dict] = {}
utils.print_update(level=1,message=f"Loading data stores for {country_name} regions with RUN_ID: {RUN_ID} and Regions: {regions}")
for region in regions:
    country_dict = {}
    try:
        store = Path(root / f"data/store/{country_kwd}/{region_code}/resources_{country_kwd}_{region}_{RUN_ID}.h5")
        assert store.exists(), f"Store path doesn't exist: {store}"

        res_data = DataHandler(store, show_structure=False)
        
        country_dict['cells'] = res_data.from_store('cells')
        country_dict['boundary'] = res_data.from_store('boundary')
        country_dict['lines'] = res_data.from_store('lines')
        country_dict['timeseries_solar'] = res_data.from_store('timeseries/solar')
        country_dict['timeseries_wind'] = res_data.from_store('timeseries/wind')
        country_dict['clusters_solar'] = res_data.from_store('clusters/solar')
        country_dict['clusters_wind'] = res_data.from_store('clusters/wind')
        country_dict['disindices_solar'] = res_data.from_store('dissolved_indices/solar')
        country_dict['disindices_wind'] = res_data.from_store('dissolved_indices/wind')
        country_dict['ts_cluster_solar'] = res_data.from_store('timeseries/clusters/solar')
        country_dict['ts_cluster_wind'] = res_data.from_store('timeseries/clusters/wind')
        
        # country_dict['di_solar'] = res_data.from_store('dissolved_indices/solar')
        # country_dict['di_wind'] = res_data.from_store('dissolved_indices/wind')                                                                 
        # country_dict['LandAvailability'] = res_data.from_store('LandAvailability')
        combined_store[region] = country_dict
        utils.print_update(level=2,message=f"✓ Loaded data for : {cfg.get('region_mapping').get(region).get('name') if cfg.get('region_mapping').get(region) else region}.") 
    except Exception as e:
        print(f"X Error with region {region}: {e}")
        continue
    res_data.show_tree(store)


## Load All Cells

In [ ]:
all_cells_dict:dict[pd.DataFrame]=[combined_store[region]['cells'] for region in combined_store]
all_cells_gdf = gpd.GeoDataFrame(pd.concat(all_cells_dict, ignore_index=False), crs=all_cells_dict[0].crs)
all_cells_gdf['ISO2']=all_cells_gdf['Province'].apply(lambda x: region_to_code[x.replace(" ","")])

- Create cells' instance for plotting (CRS-m)

In [ ]:
if all_cells_gdf.crs != CRS_m:
    all_cells_gdf_plot = all_cells_gdf.to_crs(CRS_m)
else:
    all_cells_gdf_plot = all_cells_gdf

##  Load Capacity Haircut Factors

In [ ]:
haircuts_df:pd.DataFrame=pd.read_csv(paper_resources / "data/haircuts_BC.csv")

In [ ]:
# Build dict lookups
solar_haircut_map = haircuts_df.set_index("ISO2")["solar"] / 100
wind_haircut_map  = haircuts_df.set_index("ISO2")["wind"]  / 100

In [ ]:
# Map haircut factors as explicit columns first
all_cells_gdf["solar_haircut_factor"] = all_cells_gdf["ISO2"].map(solar_haircut_map)
all_cells_gdf["wind_haircut_factor"]   = all_cells_gdf["ISO2"].map(wind_haircut_map)
# Then apply
all_cells_gdf["potential_capacity_solar_haircut"] = (
    all_cells_gdf["potential_capacity_solar"] * all_cells_gdf["solar_haircut_factor"]
)
all_cells_gdf["potential_capacity_wind_haircut"] = (
    all_cells_gdf["potential_capacity_wind"] * all_cells_gdf["wind_haircut_factor"]
)


---

# Load Validation data 

## Existing VRE sites

In [ ]:
existing_VREs_data_path=Path(root/f"data/downloaded_data/CODERS/data-pull/supply/{region_code}_wind_generators.csv")
if existing_VREs_data_path.exists():
    existing_VREs=pd.read_csv(existing_VREs_data_path)
    existing_VREs_gdf=gpd.GeoDataFrame(existing_VREs,geometry=gpd.points_from_xy(existing_VREs.longitude,existing_VREs.latitude),crs=CRS_d)
    utils.print_update(level=1,message=f"Validation data for existing VREs loaded from {existing_VREs_data_path}")
    
    existing_tech_name_mapping={
    'wind_ons':'Wind',
    'solar':'Solar'
    }
    # Create new column 'Technology' based on mapping
    existing_VREs_gdf["Technology"] = existing_VREs_gdf["gen_type"].map(existing_tech_name_mapping)

    # If some gen_type values are not in the dict, fill them with 'Unknown'
    existing_VREs_gdf["Technology"] = existing_VREs_gdf["Technology"].fillna("Unknown")

else:
    existing_VREs_gdf=None
    utils.print_warning(f"Validation data for existing VREs not found at {existing_VREs_data_path}")


if existing_VREs_gdf.crs != CRS_m:
    existing_VREs_plot = existing_VREs_gdf.to_crs(CRS_m)
else:
    existing_VREs_plot = existing_VREs_gdf

## Committed VRE Sites (BCH CFPs)

In [ ]:
committed_VREs_data_path=Path(root/"ROD_2024/BCH_CFP24.geojson")
if committed_VREs_data_path.exists():
    committed_VREs_gdf=gpd.read_file(committed_VREs_data_path, engine="fiona")
    if committed_VREs_gdf.crs is None:
        committed_VREs_gdf.set_crs(CRS_d, allow_override=True, inplace=True)
    # if committed_VREs_gdf.crs != CRS_m:
    #     committed_VREs_gdf.to_crs(CRS_m, inplace=True)
    utils.print_update(level=1,message=f"Committed VREs data loaded from {committed_VREs_data_path}")
    
    
    committed_tech_name_mapping={
        'wind':'Wind',
        'solar':'Solar'
    }
    # Create new column 'Technology' based on mapping
    committed_VREs_gdf["Technology"] = committed_VREs_gdf["resource_type"].map(existing_tech_name_mapping)

    # If some gen_type values are not in the dict, fill them with 'Unknown'
    committed_VREs_gdf["Technology"] = committed_VREs_gdf["resource_type"].fillna("Unknown")


else:
    committed_VREs_gdf=None
    utils.print_warning(f"Validation data for Committed VREs not found at {committed_VREs_data_path}")


if committed_VREs_gdf.crs != CRS_m:
    committed_VREs_plot = committed_VREs_gdf.to_crs(CRS_m)
else:
    committed_VREs_plot = committed_VREs_gdf

# Boundary

- Prepare regional boundary

In [ ]:
# Combine all region boundaries into a single GeoDataFrame
boundary_gdfs = [combined_store[region]['boundary'] for region in combined_store]
all_regions_boundary = gpd.GeoDataFrame(pd.concat(boundary_gdfs, ignore_index=True), crs=boundary_gdfs[0].crs)
all_regions_boundary_dissolved = all_regions_boundary.dissolve(by=AGGREGATION_LEVEL)[["geometry"]].reset_index()

- Process the boundary info for raster plotting

In [ ]:
all_regions_boundary_dissolved_plot=all_regions_boundary_dissolved.to_crs(CRS_m)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = all_regions_boundary_dissolved_plot.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

# Load Capacity and Scores

## Reality Checks

### Applying LCOE thresholds 

#### Define _lcoe_ thresholds and capacity haircuts

In [ ]:
solar_lcoe_threshold=90 
wind_lcoe_threshold=130

# landuse_competition_solar_share=0.50 # for example, that 50% of the land that is technically developable for solar (after all exclusions) is actually available for solar after accounting for competition with other land uses. The remaining 50% would be allocated to other uses (e.g., wind) or left undeveloped due to various constraints.
# landuse_competition_wind_share=0.50 # for example, that 50% of the land that is technically developable for wind (after all exclusions) is actually available for wind after accounting for competition with other land uses. The remaining 50% would be allocated to other uses (e.g., solar) or left undeveloped due to various constraints.

#### Isolate solar and Wind Cells

In [ ]:
solar_cells= all_cells_gdf[all_cells_gdf["lcoe_solar"] <= solar_lcoe_threshold].copy()
wind_cells= all_cells_gdf[all_cells_gdf["lcoe_wind"] <= wind_lcoe_threshold].copy()

In [ ]:
capacity_cols = ["potential_capacity_solar", "potential_capacity_wind", "potential_capacity_solar_haircut", "potential_capacity_wind_haircut"]
solar_capacity_cols = capacity_cols.copy()
wind_capacity_cols = capacity_cols.copy()
solar_aggr = get_sub_nationally_aggregated_capacity(cells_with_capacity=solar_cells, capacity_cols=solar_capacity_cols, sub_national_unit_tag=AGGREGATION_LEVEL).copy()
wind_aggr = get_sub_nationally_aggregated_capacity(cells_with_capacity=wind_cells, capacity_cols=wind_capacity_cols, sub_national_unit_tag=AGGREGATION_LEVEL).copy()


# Attributs' Map

In [ ]:
AGGREGATION_LEVEL='Region'

In [ ]:
all_regions_boundary_dissolved = all_regions_boundary.dissolve(by=AGGREGATION_LEVEL)[["geometry"]].reset_index()
all_regions_boundary_dissolved_plot=all_regions_boundary_dissolved.to_crs(CRS_m)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = all_regions_boundary_dissolved_plot.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

- plot

In [ ]:
aggregated_map=all_regions_boundary_dissolved_plot
if aggregated_map.crs != CRS_m:
    aggregated_map_plot = aggregated_map.to_crs(CRS_m)
else:
    aggregated_map_plot = aggregated_map

### Capacity

In [ ]:
MARKER_HIGHLIGHT_WIDTH

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5.5), dpi=500)

# for double coulmn figure at 500 Dpi (72points per inch) minimum wide required is 3750 px, hence figsize=(7.5, 5) is recommended
# fig.suptitle(f"Resources for {country_name}", fontsize=18, fontweight='bold')

# all_cells_gdf_plot.plot(ax=ax1, color="gray", edgecolor="none", alpha=0.7, zorder=1)
# all_cells_gdf_plot.plot(ax=ax2, color="gray", edgecolor="none", alpha=0.7, zorder=1)

vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='solar',
                datafield='capacity',
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='wind',
                datafield='capacity',
                ax=ax2, 
                show=False)

# Add existing VREs to the plot
existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf,
                                                    existing_VRE_type_column='Technology',
                                                    committed_VREs_gdf=committed_VREs_plot,
                                                    committed_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=MARKER_SCALE_EXISTING,
                                                    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
                                                    marker_scale_committed=MARKER_SCALE_COMMITTED)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf,
                                                    existing_VRE_type_column='Technology',
                                                    committed_VREs_gdf=committed_VREs_plot,
                                                    committed_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=MARKER_SCALE_EXISTING,
                                                    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
                                                    marker_scale_committed=MARKER_SCALE_COMMITTED)
# -------------------------------------------------
# Shared legend
# -------------------------------------------------
all_legends = []
all_legends.extend(solar_legends if solar_legends is not None else [])
all_legends.extend(wind_legends if wind_legends is not None else [])
# all_legends.append(no_land_patch) 

# de-duplicate legend labels
unique_handles = []
seen_labels = set()

for h in all_legends:
    label = h.get_label()
    if label not in seen_labels:
        unique_handles.append(h)
        seen_labels.add(label)

fig.legend(
    handles=unique_handles,
    loc="upper center",
    bbox_to_anchor=(0.42, 0.85),
    ncol=1,
    fontsize=9.5,
    frameon=False,
)


# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================

# # Plot only boundaries (no fill)
for ax in [ax1, ax2]:
    aggregated_map_plot.boundary.plot(ax=ax, color='k', linewidth=0.4, alpha=0.5, zorder=3)

# vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)

ax2.annotate(
    f" Scenario : {scenario_name.upper()}",
    xy=(0, 0.1), xycoords='axes fraction',
    ha='left', va='bottom',
    fontsize=9, color="#140202",
    bbox=dict(boxstyle="round,pad=0.4", edgecolor="#bbbbbb", facecolor="lightgrey", alpha=0.8)
)

plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CAPACITY.png", bbox_inches='tight')

### Capacity Factor

* Individual Maps

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5.5), dpi=1000)
all_cells_gdf_plot.plot(ax=ax1, color="gray", edgecolor="none", alpha=0.7, zorder=1)
all_cells_gdf_plot.plot(ax=ax2, color="gray", edgecolor="none", alpha=0.7, zorder=1)

vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='solar',
                datafield='CF',
                cell_edge_color='white',
                cell_linewidth=0.2,
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='wind',
                datafield='CF',
                cell_edge_color='white',
                cell_linewidth=0.2,
                ax=ax2, 
                show=False)

# Add existing VREs to the plot
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    target_crs=CRS_m,
                                                    existing_VREs_gdf=existing_VREs_gdf,
                                                    existing_VRE_type_column='Technology',
                                                    committed_VREs_gdf=committed_VREs_plot,
                                                    committed_VRE_type_column='Technology',
                                                    marker_scale_existing=MARKER_SCALE_EXISTING,
                                                    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
                                                    marker_scale_committed=MARKER_SCALE_COMMITTED)

ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf,
                                                    existing_VRE_type_column='Technology',
                                                    committed_VREs_gdf=committed_VREs_plot,
                                                    committed_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                     marker_scale_existing=MARKER_SCALE_EXISTING,
                                                    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
                                                    marker_scale_committed=MARKER_SCALE_COMMITTED)

# -------------------------------------------------
# Shared legend
# -------------------------------------------------
all_legends = []
all_legends.extend(solar_legends if solar_legends is not None else [])
all_legends.extend(wind_legends if wind_legends is not None else [])

# de-duplicate legend labels
unique_handles = []
seen_labels = set()
for h in all_legends:
    label = h.get_label()
    if label not in seen_labels:
        unique_handles.append(h)
        seen_labels.add(label)

fig.legend(
    handles=unique_handles,
    loc="upper center",
    bbox_to_anchor=(0.42, 0.85),
    ncol=1,
    fontsize=9.5,
    frameon=False,
)

# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================

# # Plot only boundaries (no fill)
for ax in [ax1, ax2]:
    aggregated_map_plot.boundary.plot(ax=ax, color='k', linewidth=0.4, alpha=0.5, zorder=3)

# vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)

ax2.annotate(
    f" Weather year : {weather_year}",
    xy=(0, 0.1), xycoords='axes fraction',
    ha='left', va='bottom',
    fontsize=9, color="#140202",
    bbox=dict(boxstyle="round,pad=0.4", edgecolor="#bbbbbb", facecolor="lightgrey", alpha=0.8)
)

# vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)
plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CF.png", bbox_inches='tight')

- plot thresholded map without haircut

## Clustering

In [ ]:
dissolved_indices_solar=combined_store['BC']['disindices_solar']
cell_ts_solar=combined_store['BC']['timeseries_solar']
dissolved_indices_wind=combined_store['BC']['disindices_wind']
cell_ts_wind=combined_store['BC']['timeseries_wind']

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def get_cluster_vs_cells_profile(region_to_plot: str, 
                                 cluster_id_to_plot: int,
                                 resource_type:str,
                                 cells_timeseries: pd.DataFrame, 
                                 dissolved_indices: pd.DataFrame, 
                                 cluster_timeseries: pd.DataFrame,
                                 plot_save_to: str | Path = None):
    """
    Plot cluster representative profile vs all member cells as daily mean ± std deviation.

    Parameters
    ----------
    region_to_plot : str
        Region name.
    cluster_id_to_plot : int
        Cluster ID.
    cells_timeseries : pd.DataFrame
        Timeseries for all cells, indexed by datetime.
    dissolved_indices : pd.DataFrame
        Mapping of cells to clusters: dissolved_indices.loc[region, cluster_id] gives list of cell IDs.
    cluster_timeseries : pd.DataFrame
        Cluster representative timeseries, indexed by datetime.
    plot_save_to : str | Path, optional
        Path to save figure. If None, figure is saved under `./vis/`.
    """
    resource_type = resource_type.lower()
    if resource_type not in ['solar', 'wind']:
        raise ValueError("resource_type must be either 'solar' or 'wind'")
    region = region_to_plot
    cluster_id = cluster_id_to_plot

    # --- 1. GET MEMBER CELLS OF THE CLUSTER ---
    cell_ids = dissolved_indices.loc[region, cluster_id]

    # --- 2. RESAMPLE TO DAILY MEAN ---
    cell_daily = [cells_timeseries[cid].resample("1D").mean() for cid in cell_ids]
    cluster_daily = cluster_timeseries[f"{region}_{cluster_id}"].resample("1D").mean()

    # --- 3. CONVERT TO 2D ARRAY (days × cells) ---
    cell_matrix = np.column_stack([s.values for s in cell_daily])

    # --- 4. CALCULATE DAILY MEAN AND STD DEV ---
    mean_cells = cell_matrix.mean(axis=1)
    std_cells = cell_matrix.std(axis=1)

    # --- 5. PLOT ---
    fig, ax = plt.subplots(figsize=(12, 3.5), dpi=1000)

    # Shaded area: ±1 std deviation
    ax.fill_between(cell_daily[0].index, mean_cells - std_cells, mean_cells + std_cells,
                    color="orange" if resource_type=='solar' else "skyblue", alpha=0.3, label="Cells ±1 Std Dev")

    # Cluster representative
    ax.plot(cluster_daily.index, cluster_daily.values, color="orangered" if resource_type=='solar' else "navy", linewidth=2, label="Cluster Profile")

    # Clean aesthetics
    ax.set_title(f"Daily Mean {resource_type.capitalize()} Profiles – {region} Cluster {cluster_id}", fontsize=14, weight="bold")
    # ax.set_xlabel("Day of Year", fontsize=12)
    ax.set_ylabel("Normalized Generation", fontsize=12)
    ax.grid(alpha=0.4,linestyle='--', linewidth=0.5)
    ax.legend(frameon=False,fontsize=13)
    ax.tick_params(axis="both", which="major", labelsize=10,labelrotation =90,direction='in',length=3)

    # Optional: make background transparent
    # fig.patch.set_alpha(0)
    # ax.set_facecolor('none')

    plt.tight_layout()

    # --- 6. SAVE FIGURE ---
    
    plot_save_to = Path(plot_save_to)/f'{resource_type}_cluster_{region}_{cluster_id}_vs_cells_profile.svg' if plot_save_to else Path(f"./vis/{resource_type}_cluster_{region}_{cluster_id}_vs_cells_profile.svg")
    plot_save_to.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(plot_save_to)
    utils.print_update(level=2, message=f"Cluster vs Cells profile plot saved to {plot_save_to}")
    plt.show()

In [ ]:
timeseries_clusters_solar=combined_store['BC']['ts_cluster_solar']
timeseries_clusters_wind=combined_store['BC']['ts_cluster_wind']

In [ ]:
get_cluster_vs_cells_profile('EastKootenay', 1, 'solar',cell_ts_solar, dissolved_indices_solar, timeseries_clusters_solar,vis_save_to_root)
get_cluster_vs_cells_profile('PeaceRiver', 1, 'wind',cell_ts_wind, dissolved_indices_wind, timeseries_clusters_wind,vis_save_to_root)

### Cells

- Method 1 (simple)

In [ ]:
def get_country_clusters_cell( resource_type:str,
                              cells: pd.DataFrame=None,
                              lcoe_threshold: float=None,
                            ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Get country-level clusters of cells that meet the LCOE thresholds for solar and wind.
    Aggregates potential capacity, area, and costs for each resource by country.
    Returns two DataFrames: one for solar and one for wind, each with aggregated metrics.
    """
    
    if lcoe_threshold is not None:
        cells = cells[cells[f'lcoe_{resource_type}']<=lcoe_threshold].copy()
    else:
        cells = cells.copy()


    cells["Country"] = cells["Country"].apply(utils.standardize_tags)
    
    agg_dict = {
        f"total_capacity_{resource_type}_afterLandUseCompetition": "sum",
        f"capex_{resource_type}": "sum",
        f"fom_{resource_type}": "sum",
        f"vom_{resource_type}": "mean",
        f"grid_connection_cost_per_km_{resource_type}": "mean",
        f"tx_line_rebuild_cost_{resource_type}": "mean",
        f"Operational_life_{resource_type}": "mean",
        f"{resource_type}_CF_mean": "mean",
        f"lcoe_{resource_type}": "mean",
    }
    clusters = cells.groupby("Country").agg(agg_dict).reset_index()
    clusters[f"total_capacity_{resource_type}_afterLandUseCompetition"] = clusters[f"total_capacity_{resource_type}_afterLandUseCompetition"].round(2)
    clusters = clusters.rename(columns={ f"total_capacity_{resource_type}_afterLandUseCompetition": f"potential_capacity_{resource_type}_MW",
                                        f"capex_{resource_type}": f"capex_{resource_type}_MilUSDperMW",
                                        f"fom_{resource_type}": f"fom_{resource_type}_MilUSDperMW",
                                        f"vom_{resource_type}": f"vom_{resource_type}_MilUSDperMW",
                                        f"grid_connection_cost_per_km_{resource_type}": f"grid_connection_cost_per_km_{resource_type}_MilUSD",
                                        f"tx_line_rebuild_cost_{resource_type}": f"tx_line_rebuild_cost_{resource_type}_MilUSD",
                                        f"Operational_life_{resource_type}": f"operational_life_{resource_type}_years",
                                        f"{resource_type}_CF_mean": f"{resource_type}_CF_mean",
                                        f"lcoe_{resource_type}": f"lcoe_{resource_type}_USD_per_MWh"
                                        })
    clusters[f"potential_capacity_{resource_type}_GW"] = (clusters[f"potential_capacity_{resource_type}_MW"] / 1000).round(3)
    return clusters

In [ ]:
solar_clusters= get_country_clusters_cell(resource_type='solar', cells=solar_combined, lcoe_threshold=solar_lcoe_threshold)
wind_clusters= get_country_clusters_cell(resource_type='wind', cells=wind_combined, lcoe_threshold=wind_lcoe_threshold)

In [ ]:
# Map Country -> Country_code
solar_clusters["Country_code"] = solar_clusters["Country"].map(region_to_code)
solar_clusters=solar_clusters.set_index(solar_clusters["Country_code"])
solar_clusters.to_csv(results_save_to_root/"solar_clusters.csv", index=False)
wind_clusters["Country_code"] = wind_clusters["Country"].map(region_to_code)
wind_clusters=wind_clusters.set_index(wind_clusters["Country_code"])
wind_clusters.to_csv(results_save_to_root/"wind_clusters.csv", index=False)
print(f"{results_save_to_root/'wind_clusters.csv'} and {results_save_to_root/'solar_clusters.csv'} saved successfully.")

### Timeseries

In [ ]:
solar_ts_dict:dict[pd.DataFrame]=[combined_store[region]['timeseries_solar'] for region in combined_store]
wind_ts_dict:dict[pd.DataFrame]=[combined_store[region]['timeseries_wind'] for region in combined_store]

In [ ]:
combined_store['BC']['timeseries_solar']

In [ ]:
solar_ts=pd.DataFrame()
wind_ts=pd.DataFrame()
for resource_type in ['solar','wind']:
    for region in combined_store.keys():
        ts_df=combined_store[region][f'timeseries_{resource_type}']
        ts_mean=ts_df.mean(axis=1) # to check if there is any non-zero value, if all values are zero then it means timeseries data is not available for that region
        ts_mean['Country']=region
        ts_mean.index.name="timestamp"
        if resource_type == 'solar':
            solar_ts[region]=ts_mean

        else:
            wind_ts[region]=ts_mean
            
            
wind_ts.to_csv(results_save_to_root/f"wind_timeseries.csv", index=True)
print(f"Saved wind timeseries to {results_save_to_root/f'wind_timeseries.csv'}")
solar_ts.to_csv(results_save_to_root/f"solar_timeseries.csv", index=True)
print(f"Saved solar timeseries to {results_save_to_root/f'solar_timeseries.csv'}")